# Supervised Classification Modeling & Tuning — Credit Default Risk

Trains and compares **4 distinct model architectures** on the preprocessed credit-default
data from the feature-engineering pipeline notebook, tunes each with
`GridSearchCV` + stratified K-fold cross-validation, evaluates all four on the same
held-out test set, and serializes the champion model to disk.

**Models compared:** Logistic Regression, Random Forest, XGBoost (falls back to
scikit-learn's Gradient Boosting automatically if `xgboost` isn't installed — see the
import cell), and an SVM with an RBF kernel.

**Dataset:** the same synthetic, seeded credit-default-risk stand-in used in the feature
engineering notebook, generated fresh here so this notebook can run standalone. Swap the
loading cell for a real CSV to use it on production data — nothing downstream needs to
change as long as the column names match.


## 1. Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay,
)
import joblib

# Prefer real XGBoost if it's installed; otherwise fall back to sklearn's own
# gradient boosting so this notebook still runs end to end with no extra installs.
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier as XGBClassifier
    HAS_XGB = False

print("Using real XGBoost:" , HAS_XGB)

pd.set_option("display.max_columns", None)
RANDOM_STATE = 7


## 2. Load data (synthetic stand-in) and split

Same generation as the feature-engineering notebook, same random seed, same
train/test split done *before* any preprocessing to prevent leakage.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 3000

age = rng.integers(18, 75, n).astype(float)
income = np.round(rng.lognormal(mean=10.5, sigma=0.5, size=n), 2)
loan_amount = np.round(rng.uniform(1000, 50000, n), 2)
credit_score = rng.integers(300, 850, n).astype(float)
debt_to_income = np.round(rng.uniform(0, 0.6, n), 3)
employment_type = rng.choice(
    ["Salaried", "Self-employed", "Unemployed", "Retired"], n, p=[0.55, 0.25, 0.1, 0.1]
).astype(object)
state = rng.choice([f"ST{i:02d}" for i in range(25)], n)

for arr, frac in [(age, 0.03), (income, 0.05), (credit_score, 0.04), (debt_to_income, 0.02)]:
    idx = rng.choice(n, int(n * frac), replace=False)
    arr[idx] = np.nan
emp_idx = rng.choice(n, int(n * 0.03), replace=False)
employment_type[emp_idx] = None

logit = (
    -1.0 - 0.01 * (credit_score - 650) + 3.0 * debt_to_income + 0.00002 * loan_amount
    - 0.01 * (income / 1000) + 0.8 * (employment_type == "Unemployed")
    + rng.normal(0, 0.7, n)
)
logit = np.nan_to_num(logit, nan=0.0)
prob_default = 1 / (1 + np.exp(-logit))
default = (rng.random(n) < prob_default).astype(int)

df = pd.DataFrame({
    "age": age, "income": income, "loan_amount": loan_amount, "credit_score": credit_score,
    "debt_to_income": debt_to_income, "employment_type": employment_type, "state": state,
    "default": default,
})

X = df.drop(columns="default")
y = df["default"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)


## 3. Preprocessing pipeline

Same `ColumnTransformer` as the feature-engineering notebook: median-imputed +
scaled numeric features, one-hot encoded low-cardinality categorical, target-encoded
high-cardinality categorical. It gets refit inside each model's pipeline below, so
every model sees identically preprocessed data with no leakage.


In [ ]:
numeric_features = ["age", "income", "loan_amount", "credit_score", "debt_to_income"]
low_card_cat = ["employment_type"]
high_card_cat = ["state"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
low_card_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
high_card_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("target_enc", TargetEncoder(random_state=RANDOM_STATE)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("low_card_cat", low_card_pipeline, low_card_cat),
    ("high_card_cat", high_card_pipeline, high_card_cat),
])


## 4. Define 4 models + hyperparameter grids

Each model is wrapped in its own `preprocess + clf` pipeline and tuned with
`GridSearchCV` using **5-fold stratified cross-validation**, scoring on ROC-AUC. Grids
are kept small so this runs in a reasonable time on a laptop — widen them if you have
more time/compute.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

model_configs = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        {"clf__C": [0.1, 1.0, 10.0]},
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=RANDOM_STATE),
        {"clf__n_estimators": [100, 300], "clf__max_depth": [4, 8, None]},
    ),
    ("XGBoost" if HAS_XGB else "Gradient Boosting (XGBoost fallback)"): (
        XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE) if HAS_XGB
        else XGBClassifier(random_state=RANDOM_STATE),
        {"clf__n_estimators": [100, 300], "clf__max_depth": [3, 5], "clf__learning_rate": [0.05, 0.1]},
    ),
    "SVM (RBF kernel)": (
        SVC(probability=True, random_state=RANDOM_STATE),
        {"clf__C": [1.0, 10.0], "clf__gamma": ["scale", "auto"]},
    ),
}
list(model_configs.keys())


## 5. Train + tune all 4 models

For each model: build the pipeline, run `GridSearchCV`, refit the best combination,
score it on the held-out test set. This cell can take a few minutes to run.


In [ ]:
results = []
fitted_pipelines = {}
roc_data = {}

for name, (estimator, param_grid) in model_configs.items():
    t0 = time.time()
    pipe = Pipeline([("preprocess", preprocessor), ("clf", estimator)])
    search = GridSearchCV(pipe, param_grid, scoring="roc_auc", cv=cv, n_jobs=-1)
    search.fit(X_train, y_train)

    best = search.best_estimator_
    proba = best.predict_proba(X_test)[:, 1]
    pred = best.predict(X_test)

    results.append({
        "model": name,
        "best_params": search.best_params_,
        "cv_roc_auc": round(search.best_score_, 4),
        "test_roc_auc": round(roc_auc_score(y_test, proba), 4),
        "test_precision": round(precision_score(y_test, pred), 4),
        "test_recall": round(recall_score(y_test, pred), 4),
        "test_f1": round(f1_score(y_test, pred), 4),
        "train_seconds": round(time.time() - t0, 1),
    })
    fitted_pipelines[name] = best
    roc_data[name] = roc_curve(y_test, proba)[:2]  # (fpr, tpr)

    print(f"Done: {name}  (test ROC-AUC = {results[-1]['test_roc_auc']}, {results[-1]['train_seconds']}s)")


## 6. Model comparison table

This is the required "model comparison table" deliverable.


In [ ]:
results_df = pd.DataFrame(results).sort_values("test_roc_auc", ascending=False).reset_index(drop=True)
results_df


## 7. ROC-AUC curves (all 4 models, overlaid)

In [ ]:
plt.figure(figsize=(7, 6))
for name, (fpr, tpr) in roc_data.items():
    auc = results_df.loc[results_df.model == name, "test_roc_auc"].values[0]
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves — model comparison")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 8. Select the champion model

Champion = highest **test-set ROC-AUC** among the four tuned models. (Swap this rule
for whatever your project prioritizes — e.g. recall, if missing a positive case is
costlier than a false alarm.)


In [ ]:
champion_name = results_df.iloc[0]["model"]
champion_pipeline = fitted_pipelines[champion_name]
print("Champion model:", champion_name)
print("Best hyperparameters:", results_df.iloc[0]["best_params"])


## 9. Confusion matrix — champion model

In [ ]:
champion_pred = champion_pipeline.predict(X_test)
cm = confusion_matrix(y_test, champion_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No default", "Default"])
disp.plot(cmap="Blues", values_format="d")
plt.title(f"Confusion matrix — {champion_name}")
plt.show()


## 10. Serialize the champion model

Saves the **entire fitted pipeline** (preprocessing + model together), so loading this
one file is enough to run predictions on new raw data — no need to separately save or
re-fit the preprocessor.


In [ ]:
joblib.dump(champion_pipeline, "champion_model.pkl")
print("Saved champion_model.pkl")

# To reuse it later:
# import joblib
# loaded = joblib.load("champion_model.pkl")
# loaded.predict(new_raw_dataframe)


## 11. Summary

- All 4 models were tuned with the same 5-fold stratified CV and scored on the same
  held-out test set, so the comparison in the table above is apples-to-apples.
- The champion (highest test ROC-AUC) is saved as `champion_model.pkl` — a single
  self-contained pipeline (preprocessing + model) ready to call `.predict()` /
  `.predict_proba()` on new raw data.
- **Caveat:** trained on synthetic data — before using this on a real decision, retrain
  on real data that satisfies the Responsible Data Card from the problem-framing stage,
  and re-check that the champion's precision/recall trade-off matches the actual
  business costs (see the problem-framing memo's false-positive/false-negative cost
  figures) rather than optimizing ROC-AUC alone.
